# Master Intent Dataset Pipeline

This notebook performs the complete End-to-End data engineering pipeline for the Intent Classification model:
1. **Load Raw AI Data**: Loads 2,400 raw questions generated by LLM (600 per class: NUTRITION_LOOKUP, HEALTH_ADVICE, BOTH, NONE).
2. **MinHash LSH Filtering**: Drops near-duplicates to ensure linguistic diversity (down to 500 per class).
3. **Rule-based Augmentation**: Generates tricky 'hard-set' examples using templates to cover edge cases.
4. **Merge & QA Check**: Combines datasets and verifies exact duplicates and class balance.
5. **Semantic Separation Check**: Uses sentence-transformers to prove the classes are semantically distinct.
6. **Export**: Saves the final intent_train_v2.csv for training.

## 1. Load Raw AI Data

In [ ]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
import os, sys

if not Path("configs/config.yaml").exists():
    os.chdir("../..")
sys.path.insert(0, ".")

raw_df = pd.read_csv('data/en/synthetic_intent.csv')
print("Raw Data Shape:", raw_df.shape)
print(raw_df['label'].value_counts())

## 2. MinHash LSH Filtering (Near-Duplicates)

In [ ]:
from datasketch import MinHash, MinHashLSH
import re

def get_minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    words = re.findall(r'\w+', str(text).lower())
    for w in words:
        m.update(w.encode('utf8'))
    return m

lsh = MinHashLSH(threshold=0.85, num_perm=128)
drop_indices = set()
seen_hashes = {}

for idx, row in raw_df.iterrows():
    m = get_minhash(row['text'])
    result = lsh.query(m)
    if result:
        drop_indices.add(idx)
    else:
        lsh.insert(idx, m)

print(f"Found {len(drop_indices)} near-duplicates.")
clean_llm_df = raw_df.drop(index=list(drop_indices)).reset_index(drop=True)

# Balance down to 500 per class
balanced_llm_df = clean_llm_df.groupby('label').apply(lambda x: x.sample(n=500, random_state=42)).reset_index(drop=True)
print("Clean LLM Data Shape:", balanced_llm_df.shape)
print(balanced_llm_df['label'].value_counts())

## 3. Rule-based Hard-set Augmentation
Generate specific tricky edge cases.

In [ ]:
import random
random.seed(42)
def unique_take(lst, n):
    return random.sample(list(set(lst)), n)

FOODS = ["egg", "chicken breast", "apple", "salmon", "spinach", "brown rice", "almond", "milk", "broccoli", "beef", "orange", "banana", "yogurt", "peanut butter", "oats", "sweet potato", "cheese", "carrot", "avocado", "lentils", "tofu", "kale", "blueberries", "tomato", "walnuts", "chia seeds", "quinoa", "black beans", "olive oil", "garlic"]
NUTRIENTS = ["protein", "calorie", "fat", "carb", "vitamin C", "iron", "calcium", "fiber", "sugar", "sodium", "potassium", "magnesium", "zinc", "vitamin D", "cholesterol"]
CONDITIONS = ["diabetes", "hypertension", "obesity", "kidney disease", "insomnia", "acid reflux", "celiac disease", "gout", "anemia", "osteoporosis", "IBS", "cholesterol", "arthritis", "fatty liver", "pregnancy"]
HERBS = ["ginger", "garlic", "ginseng", "cinnamon", "turmeric", "hibiscus", "saffron", "soy", "aloe", "mint", "chamomile", "lavender", "echinacea", "rosemary", "basil"]

hard_rows = []
# Nutrition Hard
t_nut = []
for f in FOODS:
    for n in NUTRIENTS:
        t_nut.append(f"How much {n} is in {f} exactly?")
        t_nut.append(f"Give me the {n} value for a portion of {f}.")
hard_rows.extend([{'text': t, 'label': 'NUTRITION_LOOKUP'} for t in unique_take(t_nut, 60)])

# Health Hard (From herbs/conditions)
t_health = []
for h in HERBS:
    for c in CONDITIONS:
        t_health.append(f"Is {h} safe for someone with {c}?")
        t_health.append(f"Can {h} help treat {c} symptoms?")
hard_rows.extend([{'text': t, 'label': 'HEALTH_ADVICE'} for t in unique_take(t_health, 60)])

# Both Hard
t_both = []
for f in FOODS:
    for n in NUTRIENTS:
        for c in CONDITIONS:
            t_both.append(f"How much {n} is in {f}, and is it safe for {c}?")
hard_rows.extend([{'text': t, 'label': 'BOTH'} for t in unique_take(t_both, 60)])

# None Hard
t_none = []
for c in ["Tokyo", "London", "New York", "Paris", "Berlin", "Dubai", "Sydney", "Rome", "Seoul", "Toronto"]:
    t_none.append(f"What is the current population of {c}?")
    t_none.append(f"Tell me the history of {c}.")
    t_none.append(f"How far is {c} from here?")
    t_none.append(f"Can you guide me to {c}?")
    t_none.append(f"What language is spoken in {c}?")
    t_none.append(f"Show me a map of {c}.")
hard_rows.extend([{'text': t, 'label': 'NONE'} for t in unique_take(t_none, 60)])

hard_df = pd.DataFrame(hard_rows)
print("Hard Set Shape:", hard_df.shape)
print(hard_df['label'].value_counts())

## 4. Merge & QA Check

In [ ]:
final_df = pd.concat([balanced_llm_df, hard_df], ignore_index=True)

# Shuffle
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final Dataset Shape:", final_df.shape)
counts = final_df['label'].value_counts()
print(counts)
print("\nBalanced:", counts.nunique() == 1)

exact_dupes = final_df.duplicated("text").sum()
print("Exact duplicate questions:", exact_dupes)

if exact_dupes > 0:
    final_df = final_df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    print("Dropped exact duplicates. New shape:", final_df.shape)

## 5. Semantic Separation Check
Verify that the 4 intent classes are semantically distinct using embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Loading SentenceTransformer...")
model = SentenceTransformer('all-MiniLM-L6-v2')

sample_df = final_df.groupby('label').apply(lambda x: x.sample(n=100, random_state=42)).reset_index(drop=True)
texts = sample_df['text'].tolist()
labels = sample_df['label'].tolist()

print("Computing embeddings...")
embeddings = model.encode(texts, show_progress_bar=True)
sim_matrix = cosine_similarity(embeddings)

within_sims = []
between_sims = []

for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        if labels[i] == labels[j]:
            within_sims.append(sim_matrix[i, j])
        else:
            between_sims.append(sim_matrix[i, j])

avg_within = np.mean(within_sims)
avg_between = np.mean(between_sims)

print(f"Avg within-class similarity:  {avg_within:.3f}")
print(f"Avg between-class similarity: {avg_between:.3f}")
print(f"Separation ratio: {avg_within/avg_between:.2f}x")

## 6. Export Final Dataset

In [ ]:
export_path = Path("data/en/intent_v2/intent_train_v2.csv")
export_path.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(export_path, index=False)
print(f"Successfully exported {len(final_df)} rows to {export_path}")